In [1]:
import statistics

import pandas as pd

from helpers import *
from regression import *

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
CONFIG = {
    'elo_date': '2025-11-30',  # date for elo ratings from clubelo.com
    'opta_date': '2025-11-28',  # date for elo ratings from the opta -> clubelo regression; elo_date from day X is before the games are played, for opta it depends
    'number_of_sims': 10000,
    'league_id': 106,
    'season': 2025,
    'head_size': 36,
    'country_code_elo': None,  # use this attr. to use elo ratings from clubelo.com
    'country_code_api': 'POL',  # use this attr. to use elo ratings from the opta -> clubelo regression
    'stdev': 0,
    'update_fixtures': True,
    'is_european_league': False,
    'round_no': 18,
}
code = CONFIG['country_code_elo'] if CONFIG['country_code_elo'] is not None else CONFIG['country_code_api']
CONFIG['sorting_order'] = get_sorting_order_for_country_code(code)

In [5]:
# download_elo_data(CONFIG['elo_date'])

In [6]:
# main_regression(**CONFIG)

In [7]:
standings_df = build_historical_standings_table_after_at_most_n_rounds(**CONFIG)
standings_df.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Points,Random order
1,Wisla Plock,1443.38,18,7,9,2,21,12,9,30,11
2,Gornik Zabrze,1513.38,18,9,3,6,29,24,5,30,8
3,Raków Częstochowa,1549.50,17,9,2,6,26,22,4,29,2
4,Jagiellonia,1529.18,16,8,4,4,28,20,8,28,1
5,Cracovia Krakow,1481.76,18,7,6,5,25,21,4,27,3
6,Radomiak Radom,1420.80,18,7,5,6,35,30,5,26,9
7,Lech Poznan,1520.15,17,6,8,3,29,26,3,26,14
8,Zaglebie Lubin,1414.02,17,6,7,4,30,24,6,25,0
9,Korona Kielce,1452.41,18,6,6,6,21,19,2,24,5
10,Pogon Szczecin,1465.96,18,6,3,9,28,32,-4,21,12


In [9]:
with open('data/optimized_elos_2025_2026.json', 'r') as f:
    optimized_elos = json.load(f)

team_map = pd.read_excel('teams_mapping/team_names.xlsx')
team_map = team_map[['fixtures_name', 'football-data_name']]
team_map.dropna(inplace=True)
team_map_dict = dict(zip(team_map['football-data_name'], team_map['fixtures_name']))
optimized_elos_mapped = {
    team_map_dict.get(team, team): elo for team, elo in optimized_elos.items()
}
standings_df['Elo'] = standings_df['Club'].map(optimized_elos_mapped)
standings_df.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Points,Random order
1,Wisla Plock,1483.49,18,7,9,2,21,12,9,30,11
2,Gornik Zabrze,1551.61,18,9,3,6,29,24,5,30,8
3,Raków Częstochowa,1605.27,17,9,2,6,26,22,4,29,2
4,Jagiellonia,1552.33,16,8,4,4,28,20,8,28,1
5,Cracovia Krakow,1521.13,18,7,6,5,25,21,4,27,3
6,Radomiak Radom,1470.72,18,7,5,6,35,30,5,26,9
7,Lech Poznan,1585.70,17,6,8,3,29,26,3,26,14
8,Zaglebie Lubin,1445.62,17,6,7,4,30,24,6,25,0
9,Korona Kielce,1522.17,18,6,6,6,21,19,2,24,5
10,Pogon Szczecin,1521.29,18,6,3,9,28,32,-4,21,12


In [10]:
CONFIG['update_fixtures'] = False

In [11]:
float(round(standings_df['Points'].sum() / standings_df['Matches played'].sum(), 2))

1.34

In [12]:
sample_season = simulate_season_after_n_rounds(**CONFIG, standings_df=standings_df)
sample_season.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Points,Random order
1,Cracovia Krakow,1521.13,34,18,8,8,49,29,20,62,13
2,Jagiellonia,1552.33,34,17,8,9,50,34,16,59,8
3,Raków Częstochowa,1605.27,34,18,5,11,47,35,12,59,12
4,Wisla Plock,1483.49,34,13,14,7,38,27,11,53,4
5,Legia Warszawa,1625.11,34,14,10,10,42,31,11,52,6
6,Korona Kielce,1522.17,34,14,9,11,40,32,8,51,10
7,Lech Poznan,1585.70,34,12,14,8,47,42,5,50,17
8,Gornik Zabrze,1551.61,34,14,5,15,41,44,-3,47,2
9,Radomiak Radom,1470.72,34,12,10,12,50,47,3,46,11
10,Zaglebie Lubin,1445.62,34,11,11,12,44,44,0,44,9


In [13]:
float(round(sample_season['Points'].sum() / sample_season['Matches played'].sum(), 2))

1.36

In [14]:
simulate_odds(**CONFIG, standings_df=standings_df).head(CONFIG['head_size'])

,Home Team,Away Team,Odds H,Odds D,Odds A
0,Lechia Gdansk,Gornik Zabrze,2.74,3.43,2.91
1,Arka Gdynia,Motor Lublin,2.74,3.43,2.91
2,Pogon Szczecin,Radomiak Radom,2.06,3.72,4.08
3,Zaglebie Lubin,Widzew Łódź,3.03,3.44,2.64
4,Piast Gliwice,Legia Warszawa,3.27,3.47,2.46
5,Nieciecza,Jagiellonia,3.53,3.54,2.30
6,Cracovia Krakow,Lech Poznan,2.79,3.42,2.86
7,Raków Częstochowa,GKS Katowice,1.69,4.39,5.56
8,Korona Kielce,Wisla Plock,2.12,3.66,3.93


In [15]:
# full table sim
results = run_full_table_sims(**CONFIG, standings_df=standings_df)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:32<00:00, 310.36it/s]


10000 simulations


,Club,Elo,xPts,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
1,Raków Częstochowa,1605.27,57.82,37.7,21.8,14.0,9.2,6.4,3.9,2.5,1.8,1.1,0.7,0.4,0.2,0.1,0.1,0.1,0.0,0.0,0.0
2,Jagiellonia,1552.33,55.34,23.2,20.1,15.2,11.8,8.4,6.3,4.7,3.2,2.4,1.8,1.2,0.7,0.5,0.2,0.1,0.1,0.0,0.0
3,Gornik Zabrze,1551.61,53.93,13.8,15.7,15.6,13.8,11.5,8.6,6.6,4.9,3.3,2.3,1.4,1.1,0.7,0.4,0.2,0.1,0.0,0.0
4,Lech Poznan,1585.70,53.26,12.2,14.9,15.0,13.6,10.7,8.5,6.8,5.7,4.0,2.9,2.0,1.6,0.9,0.7,0.3,0.1,0.1,0.0
5,Wisla Plock,1483.49,50.08,3.9,7.5,10.0,10.7,11.2,10.9,10.4,8.7,7.5,5.8,4.3,3.5,2.5,1.6,0.9,0.6,0.2,0.1
6,Cracovia Krakow,1521.13,49.62,3.3,6.3,8.5,9.6,10.9,11.4,10.1,9.1,7.9,6.1,5.2,4.0,2.8,2.2,1.2,0.8,0.4,0.1
7,Legia Warszawa,1625.11,48.96,3.3,6.0,8.3,10.1,10.7,10.4,10.3,9.0,7.4,6.1,5.2,4.1,3.4,2.2,1.5,1.1,0.5,0.4
8,Korona Kielce,1522.17,46.47,1.0,2.4,3.9,5.5,6.7,8.5,9.3,9.6,9.2,8.7,8.5,7.1,6.2,4.9,3.7,2.9,1.4,0.5
9,Radomiak Radom,1470.72,45.78,0.8,2.0,3.3,4.7,6.3,7.4,8.2,8.8,9.9,9.6,8.5,7.7,6.7,5.8,4.4,3.1,2.1,0.8
10,Zaglebie Lubin,1445.62,44.18,0.5,1.1,2.0,3.0,4.3,5.5,6.5,7.5,8.8,9.1,9.3,9.3,8.3,7.8,6.5,5.3,3.5,1.6


In [16]:
# top 1
winning_places = 1
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=False)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_top_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:28<00:00, 346.92it/s]

10000 simulations
1 winning places


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Raków Częstochowa,1605.27,57.71,3613,18,593,36.1,36.3,42.2,2.77,2.75,2.37
2,Jagiellonia,1552.33,55.43,2025,347,96,20.2,23.7,24.7,4.94,4.22,4.05
3,Gornik Zabrze,1551.61,53.91,1249,115,251,12.5,13.6,16.2,8.01,7.33,6.19
4,Lech Poznan,1585.70,53.26,1056,189,146,10.6,12.4,13.9,9.47,8.03,7.19
5,Wisla Plock,1483.49,50.10,329,107,27,3.3,4.4,4.6,30,23,22
6,Cracovia Krakow,1521.13,49.59,255,85,37,2.6,3.4,3.8,39,29,27
7,Legia Warszawa,1625.11,48.85,229,99,1,2.3,3.3,3.3,44,30,30
8,Korona Kielce,1522.17,46.36,66,29,12,0.7,1.0,1.1,152,105,93
9,Radomiak Radom,1470.72,45.79,50,30,4,0.5,0.8,0.8,200,125,119
10,Zaglebie Lubin,1445.62,44.22,34,12,1,0.3,0.5,0.5,294,217,213


In [17]:
# top 2
winning_places = 2
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=False)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_top_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:29<00:00, 344.51it/s]

10000 simulations
2 winning places


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Raków Częstochowa,1605.27,57.88,5891,16,579,58.9,59.1,64.9,1.7,1.69,1.54
2,Jagiellonia,1552.33,55.44,4038,388,247,40.4,44.3,46.7,2.48,2.26,2.14
3,Gornik Zabrze,1551.61,54.06,2893,128,459,28.9,30.2,34.8,3.46,3.31,2.87
4,Lech Poznan,1585.70,53.14,2391,294,267,23.9,26.8,29.5,4.18,3.72,3.39
5,Wisla Plock,1483.49,49.96,885,240,92,8.8,11.2,12.2,11,8.89,8.22
6,Cracovia Krakow,1521.13,49.72,818,155,130,8.2,9.7,11.0,12,10,9.07
7,Legia Warszawa,1625.11,48.92,622,230,8,6.2,8.5,8.6,16,12,12.00
8,Korona Kielce,1522.17,46.27,234,85,20,2.3,3.2,3.4,43,31,30.00
9,Radomiak Radom,1470.72,45.77,188,61,9,1.9,2.5,2.6,53,40,39.00
10,Zaglebie Lubin,1445.62,44.19,123,43,3,1.2,1.7,1.7,81,60,59.00


In [18]:
# top 3
winning_places = 3
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=False)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_top_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:29<00:00, 340.16it/s]


10000 simulations
3 winning places


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Raków Częstochowa,1605.27,57.83,7263,15,526,72.6,72.8,78.0,1.38,1.37,1.28
2,Jagiellonia,1552.33,55.48,5654,343,299,56.5,60.0,63.0,1.77,1.67,1.59
3,Gornik Zabrze,1551.61,53.94,4371,104,603,43.7,44.8,50.8,2.29,2.23,1.97
4,Lech Poznan,1585.70,53.21,3922,257,398,39.2,41.8,45.8,2.55,2.39,2.18
5,Wisla Plock,1483.49,50.02,1762,341,153,17.6,21.0,22.6,5.68,4.76,4.43
6,Cracovia Krakow,1521.13,49.69,1590,247,211,15.9,18.4,20.5,6.29,5.44,4.88
7,Legia Warszawa,1625.11,48.94,1339,401,28,13.4,17.4,17.7,7.47,5.75,5.66
8,Korona Kielce,1522.17,46.28,500,150,79,5.0,6.5,7.3,20.00,15.00,14.00
9,Radomiak Radom,1470.72,45.78,463,153,28,4.6,6.2,6.4,22.00,16.00,16.00
10,Zaglebie Lubin,1445.62,44.14,250,107,22,2.5,3.6,3.8,40.00,28.00,26.00


In [19]:
# top 4
winning_places = 4
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=False)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_top_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:29<00:00, 340.35it/s]

10000 simulations
4 winning places


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Raków Częstochowa,1605.27,57.84,8321,16,354,83.2,83.4,86.9,1.20,1.20,1.15
2,Jagiellonia,1552.33,55.43,6791,219,310,67.9,70.1,73.2,1.47,1.43,1.37
3,Gornik Zabrze,1551.61,53.93,5830,70,554,58.3,59.0,64.5,1.72,1.69,1.55
4,Lech Poznan,1585.70,53.16,5242,230,450,52.4,54.7,59.2,1.91,1.83,1.69
5,Wisla Plock,1483.49,50.06,2787,360,213,27.9,31.5,33.6,3.59,3.18,2.98
6,Cracovia Krakow,1521.13,49.57,2552,263,313,25.5,28.2,31.3,3.92,3.55,3.20
7,Legia Warszawa,1625.11,48.82,2260,456,47,22.6,27.2,27.6,4.42,3.68,3.62
8,Korona Kielce,1522.17,46.43,1016,239,116,10.2,12.6,13.7,9.84,7.97,7.29
9,Radomiak Radom,1470.72,45.77,839,237,52,8.4,10.8,11.3,12.00,9.29,8.87
10,Zaglebie Lubin,1445.62,44.26,491,158,35,4.9,6.5,6.8,20.00,15.00,15.00


In [20]:
# bottom 3
winning_places = 3
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=True)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_bottom_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:29<00:00, 343.71it/s]

10000 simulations
3 winning places
Reverse: TRUE


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Nieciecza,1414.30,36.08,6336,541,108,63.4,68.8,69.8,1.58,1.45,1.43
2,Arka Gdynia,1409.67,37.49,5150,704,0,51.5,58.5,58.5,1.94,1.71,1.71
3,Motor Lublin,1462.83,40.33,3015,414,233,30.2,34.3,36.6,3.32,2.92,2.73
4,GKS Katowice,1470.22,40.94,2645,353,212,26.4,30.0,32.1,3.78,3.34,3.12
5,Lechia Gdansk,1495.00,41.03,2516,23,582,25.2,25.4,31.2,3.97,3.94,3.20
6,Piast Gliwice,1507.55,41.22,2448,77,507,24.5,25.2,30.3,4.08,3.96,3.30
7,Widzew Łódź,1535.02,42.86,1568,104,333,15.7,16.7,20.0,6.38,5.98,4.99
8,Pogon Szczecin,1521.29,43.24,1353,180,221,13.5,15.3,17.5,7.39,6.52,5.70
9,Zaglebie Lubin,1445.62,44.32,902,92,276,9.0,9.9,12.7,11,10.00,7.87
10,Radomiak Radom,1470.72,45.72,533,63,140,5.3,6.0,7.4,19,17.00,14.00


In [21]:
# bottom 1
winning_places = 1
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=True)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_bottom_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:29<00:00, 338.99it/s]

10000 simulations
1 winning places
Reverse: TRUE


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Nieciecza,1414.30,35.96,3137,375,234,31.4,35.1,37.5,3.19,2.85,2.67
2,Arka Gdynia,1409.67,37.59,1887,513,0,18.9,24.0,24.0,5.3,4.17,4.17
3,Motor Lublin,1462.83,40.31,866,121,178,8.7,9.9,11.6,12,10,8.58
4,GKS Katowice,1470.22,40.98,713,84,152,7.1,8.0,9.5,14,13,11
5,Piast Gliwice,1507.55,41.09,712,12,215,7.1,7.2,9.4,14,14,11
6,Lechia Gdansk,1495.00,41.09,586,4,220,5.9,5.9,8.1,17,17,12
7,Widzew Łódź,1535.02,42.86,303,21,128,3.0,3.2,4.5,33,31,22
8,Pogon Szczecin,1521.29,43.19,289,28,93,2.9,3.2,4.1,35,32,24
9,Zaglebie Lubin,1445.62,44.25,151,12,66,1.5,1.6,2.3,66,61,44
10,Radomiak Radom,1470.72,45.75,68,7,34,0.7,0.8,1.1,147,133,92
